# Health Record Rights Index: a starter notebook

The index scores 198 countries and territories from 0 to 100 on one question: does a person get to see, control and share their own health record? This notebook loads the scores CSV with pandas, then asks the JSON API for the same data and for two US states side by side.

License: CC BY 4.0. Credit: **SuperTruth, Inc., Health Record Rights Index**, with a link to https://healthrecordrights.com/. No key is needed. The API allows 60 requests a minute per address, with bursts up to 120.

## 1. The scores CSV

In [1]:
import pandas as pd
import requests

SITE = "https://healthrecordrights.com/"
API = SITE + "api/v1/"

scores = pd.read_csv(SITE + "data/who-holds-the-record-scores.csv")
print(scores.shape)
scores.head()

(198, 21)


,iso3,country,region,overall,rank,likely_rank,band,keys_model,confidence,dti_evidence,...,access,control,privacy,journey,commercial,clinical,research,ai,stories,as_of
0,FIN,Finland,Europe,71,1,1 to 6,Strong,Shared,high,92,...,76,74,62,82,68,76,58,50,5,2026-10-02
1,DNK,Denmark,Europe,70,2,1 to 8,Strong,Shared,medium,87,...,78,70,62,82,58,78,49,50,5,2026-10-02
2,EST,Estonia,Europe,68,3=,1 to 13,Strong,Shared,medium,88,...,80,68,55,80,58,75,50,50,4,2026-10-02
3,HUN,Hungary,Europe,68,3=,1 to 13,Strong,Shared,high,88,...,72,72,62,78,55,76,47,54,5,2026-10-01
4,SWE,Sweden,Europe,67,5,1 to 17,Strong,Shared,high,89,...,77,72,64,66,60,64,49,50,5,2026-10-02


Each row is one country or territory. `overall` is the score out of 100. `likely_rank` is the range a country could rank in, because scores move in steps of about 5 points: always quote a rank with that range. The countries at the top are a lead group, never one winner, so do not call any one of them first or best (the API marks them with `leadGroup`, below).

In [2]:
scores["overall"].describe()

count    198.000000
mean      41.898990
std       12.676766
min       19.000000
25%       33.000000
50%       39.000000
75%       50.750000
max       71.000000
Name: overall, dtype: float64

## 2. How many countries fall in each band

In [3]:
band_order = ["Leading", "Strong", "Mixed", "Weak", "Poor"]
scores["band"].value_counts().reindex(band_order, fill_value=0)

band
Leading      0
Strong       9
Mixed       66
Weak       111
Poor        12
Name: count, dtype: int64

## 3. The eight rights

Each country has a score out of 100 for each of eight rights. The overall score is their weighted average (the weights are in `/api/v1/method`).

In [4]:
rights = ["access", "control", "privacy", "journey", "commercial", "clinical", "research", "ai"]
scores[rights].describe().loc[["mean", "50%", "min", "max"]].round(1)

,access,control,privacy,journey,commercial,clinical,research,ai
mean,45.4,38.1,43.7,42.5,42.8,38.5,47.5,35.4
50%,42.5,30.0,45.0,38.0,45.0,33.5,46.0,30.0
min,15.0,20.0,15.0,18.0,15.0,15.0,30.0,15.0
max,80.0,74.0,65.0,82.0,70.0,78.0,70.0,66.0


In [5]:
# The typical (median) score for each right, lowest first
scores[rights].median().sort_values()

control       30.0
ai            30.0
clinical      33.5
journey       38.0
access        42.5
privacy       45.0
commercial    45.0
research      46.0
dtype: float64

## 4. The same data from the API

In [6]:
r = requests.get(API + "countries", timeout=30)
r.raise_for_status()
body = r.json()
print("Data as of", body["asOf"], "| version", body["version"], "| credit:", body["license"]["credit"])
countries = pd.DataFrame(body["data"]["countries"])
countries[["iso3", "name", "overall", "band", "rank", "leadGroup", "likelyRankText", "confidence"]].head(10)

Data as of 2026-10-02 | version 1.0 | credit: SuperTruth, Inc., Health Record Rights Index


,iso3,name,overall,band,rank,leadGroup,likelyRankText,confidence
0,DNK,Denmark,70,Strong,NaN,True,1 to 8,medium
1,EST,Estonia,68,Strong,NaN,True,1 to 13,medium
2,FIN,Finland,71,Strong,NaN,True,1 to 6,high
3,HUN,Hungary,68,Strong,NaN,True,1 to 13,high
4,SWE,Sweden,67,Strong,NaN,True,1 to 17,high
5,AUS,Australia,66,Strong,6.0,False,2 to 20,medium
6,FRA,France,66,Strong,6.0,False,2 to 20,high
7,NOR,Norway,66,Strong,6.0,False,2 to 20,high
8,AUT,Austria,65,Strong,9.0,False,3 to 23,high
9,PRT,Portugal,64,Mixed,10.0,False,3 to 26,high


Lead-group countries have no rank number (`rank` is empty), on purpose: they are a group. Every other rank comes with `likelyRankText`.

In [7]:
countries.loc[countries["leadGroup"], ["name", "overall", "likelyRankText"]]

,name,overall,likelyRankText
0,Denmark,70,1 to 8
1,Estonia,68,1 to 13
2,Finland,71,1 to 6
3,Hungary,68,1 to 13
4,Sweden,67,1 to 17


## 5. One country in detail

In [8]:
fin = requests.get(API + "countries/FIN", timeout=30).json()["data"]
print(f"{fin['name']}: {fin['overall']} out of 100, {fin['band']} band, {fin['rankText']}")
pd.DataFrame([{"right": c["name"], "weight": c["weight"], "score": c["score"]} for c in fin["categories"].values()])

Finland: 71 out of 100, Strong band, one of five countries that lead the 198 countries and territories we rated


,right,weight,score
0,Getting your full record,20,76
1,Control over who sees it,20,74
2,Privacy and security,15,62
3,One record across your care,15,82
4,Protection from sale and ads,10,68
5,Doctors see it when treating you,10,76
6,Your say over research use,5,58
7,Rules for AI in your care,5,50


## 6. Two US states side by side

There is no state score, total or rank. Each row says whether the two states' answers differ. DC is not a state.

In [9]:
cmp = requests.get(API + "states/compare", params={"s": "CA,TX"}, timeout=30).json()["data"]
print(cmp["rule"])
rows = [{"question": row["question"], "comparison": row["comparison"],
         **{a["usps"]: a["short"] if a["checked"] else "not checked" for a in row["answers"]}}
        for row in cmp["rows"]]
pd.DataFrame(rows)

Columns are in the order given. Each row says differs, same or cannot_compare (an answer was not checked; not checked is not a no). There is no total, no rank and no winner: the questions are on different scales, and some have no best answer.


,question,comparison,CA,TX
0,Do apps and wearables need your OK before shar...,same,"Opt-in, for some businesses","Opt-in, for some businesses"
1,Do you get a copy faster or free in some cases?,same,"Faster, and free in some cases","Faster, and free in some cases"
2,Can you say no to sharing through a health inf...,same,No general choice,No general choice
3,"Are some sensitive records, like mental health...",same,Has a rule,Has a rule
4,"Are some sensitive records, like mental health...",same,Has a rule,Has a rule
5,"Are some sensitive records, like mental health...",same,Has a rule,Has a rule
6,"Are some sensitive records, like mental health...",same,Has a rule,Has a rule
7,Can you sue if your records are misused?,same,"Yes, in general","Yes, in general"
8,Must you be told if health data leaks in a bre...,same,Medical and insurance data,Medical and insurance data
9,Are there rules for AI used in your care?,differs,3 of 3 rules,2 of 3 rules


## 7. How to cite

In [10]:
print(body["citation"])
print("License:", body["license"]["url"], "| credit:", body["license"]["credit"])

Snyder, J. A., Hill, R. P., IV, Raney, D., & Sims, L. (2026). Health Record Rights Index (version 1.0). SuperTruth Inc. https://healthrecordrights.com/ https://doi.org/10.5281/zenodo.23120174
License: https://creativecommons.org/licenses/by/4.0/ | credit: SuperTruth, Inc., Health Record Rights Index
